# 04 — Heightmap Generation
Compare synthetic (gradient-based) vs. diffusion-based tactile heightmap generation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random

In [ ]:
# ── Load a sample texture ──
dtd = Path('../data/dtd/images')
if dtd.exists():
    cats = sorted(dtd.iterdir())
    sample_path = str(random.choice(list(cats[5].glob('*.jpg'))))
else:
    print('DTD not found — set sample_path manually')
    sample_path = 'texture.jpg'

texture = Image.open(sample_path).convert('RGB')
print(f'Loaded: {sample_path}')
plt.figure(figsize=(4,4))
plt.imshow(texture); plt.title('Source Texture'); plt.axis('off'); plt.show()

In [ ]:
# ── Synthetic heightmap — parameter sweep ──
from data.heightmap_generator import heightmap_from_path

sigmas = [0.5, 1.0, 2.0, 4.0, 8.0]
fig, axes = plt.subplots(2, len(sigmas), figsize=(18, 6))

for i, sigma in enumerate(sigmas):
    hm = heightmap_from_path(sample_path, sigma=sigma)
    hm_arr = np.array(hm)
    axes[0][i].imshow(hm_arr, cmap='gray')
    axes[0][i].set_title(f'σ={sigma}')
    axes[0][i].axis('off')
    
    axes[1][i].imshow(hm_arr, cmap='hot')
    axes[1][i].axis('off')

axes[0][0].set_ylabel('Grayscale', fontsize=10)
axes[1][0].set_ylabel('Heat map', fontsize=10)
plt.suptitle('Synthetic Heightmaps at Different Gaussian Blur Sigma Values', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── Heightmap statistics per DTD category ──
from data.heightmap_generator import heightmap_from_path

if dtd.exists():
    stats = {}
    sample_cats = [c.name for c in sorted(dtd.iterdir())[:10]]
    for cat in sample_cats:
        imgs = list((dtd / cat).glob('*.jpg'))[:5]
        means = [np.array(heightmap_from_path(str(p), sigma=2.0)).mean() for p in imgs]
        stds  = [np.array(heightmap_from_path(str(p), sigma=2.0)).std()  for p in imgs]
        stats[cat] = {'mean': np.mean(means), 'std': np.mean(stds)}

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(stats.keys(), [v['mean'] for v in stats.values()], color='steelblue')
    axes[0].set_xticklabels(list(stats.keys()), rotation=45, ha='right')
    axes[0].set_ylabel('Mean height value'); axes[0].set_title('Mean Height per Category')

    axes[1].bar(stats.keys(), [v['std'] for v in stats.values()], color='salmon')
    axes[1].set_xticklabels(list(stats.keys()), rotation=45, ha='right')
    axes[1].set_ylabel('Std height value'); axes[1].set_title('Height Std Dev per Category')
    plt.tight_layout(); plt.show()

In [ ]:
# ── Diffusion-based heightmap (if model available) ──
try:
    from models.heightmap_diffusion import HeightmapDiffusionPipeline
    import torch
    
    LORA_PATH = '../checkpoints/heightmap_lora/final'
    lora_path = LORA_PATH if Path(LORA_PATH).exists() else None
    
    pipeline = HeightmapDiffusionPipeline(lora_weights_path=lora_path)
    hm_diffusion = pipeline.generate(texture, strength=0.8, num_inference_steps=30, seed=42)
    pipeline.unload()
    
    hm_synthetic = heightmap_from_path(sample_path, sigma=2.0)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(texture); axes[0].set_title('Input Texture'); axes[0].axis('off')
    axes[1].imshow(np.array(hm_synthetic), cmap='gray'); axes[1].set_title('Synthetic (Sobel)'); axes[1].axis('off')
    axes[2].imshow(np.array(hm_diffusion), cmap='gray'); axes[2].set_title('Diffusion (SD+LoRA)'); axes[2].axis('off')
    plt.suptitle('Heightmap Generation Comparison'); plt.tight_layout(); plt.show()
    
except Exception as e:
    print(f'Diffusion model not available: {e}')
    print('Run training/train_heightmap.py first, or use --method synthetic')

In [ ]:
# ── 3D surface plot of heightmap ──
from mpl_toolkits.mplot3d import Axes3D

hm = heightmap_from_path(sample_path, sigma=2.0)
hm_arr = np.array(hm.resize((64, 64))).astype(np.float32) / 255.0

x = np.arange(0, 64)
y = np.arange(0, 64)
X, Y = np.meshgrid(x, y)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, hm_arr, cmap='terrain', linewidth=0, antialiased=True)
fig.colorbar(surf, shrink=0.4, label='Height')
ax.set_title('3D Surface Visualization of Generated Heightmap')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Height')
plt.tight_layout(); plt.show()